# Inspeção visual: HiLo no FGADR (sufixo _3, conjunto de teste)

Run: `outputs/multi4sig/all4-HL0_hilo_idwt_L1_allskips_sigmoid/seed42` (EfficientNet-B4 UNet,
HiLo nas 4 skips, sigmoid, 4 classes).

- Split de teste idêntico ao do `train.py` (`outputs/cv_splits_3.json`, 89 imagens).
- Predição: média dos **logits** dos 5 folds (ensemble, sem TTA), sigmoid, limiar 0.5, igual à
  avaliação de teste do `train.py`. Com `FOLD = k` você usa só um modelo.
- O carregamento das imagens replica o `dataset.py` em cv2 puro (resize 512, normalização ImageNet),
  para não importar o `dataset.py`, que puxa o sklearn.
- O Dice por imagem daqui serve só para **escolher imagens para olhar**. Não é métrica de decisão.

Rode as células em ordem. As células de amostragem aleatória podem ser re-executadas à vontade.

In [ ]:
import os, sys, json, random, dataclasses
from pathlib import Path

import cv2                      # antes do torch (conflito de CXXABI)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path('/home/lucas/mestrado/segmentation_retinopathy')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from config import Config
from models import build_model

RUN_DIR = REPO / 'outputs/multi4sig/all4-HL0_hilo_idwt_L1_allskips_sigmoid/seed42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
THR = 0.5      # limiar da avaliação do train.py
FOLD = None    # None = ensemble dos 5 folds; 0..4 = um fold só
LIMIT = None   # corta o conj. de teste na tabela de Dice (debug); None = todas

plt.rcParams['figure.dpi'] = 110
print('device:', DEVICE)

## Config e split de teste

In [ ]:
rc = json.loads((RUN_DIR / 'run_config.json').read_text())
fields = {f.name for f in dataclasses.fields(Config)}
config = Config(**{k: tuple(v) if isinstance(v, list) else v
                   for k, v in rc['config'].items() if k in fields})
CLASSES = list(config.classes)
CODES = list(config.class_codes)
H, W = config.image_size
print(rc['exp_id'], rc['exp_name'], '| classes:', CLASSES, '| fusion:', config.wavelet_fusion,
      '| skips:', config.wavelet_skip_indices, '| encoder:', config.encoder_name)

# mesmo DataFrame e mesma ordem do train.py: presence csv + grau DR, filtro de sufixo, iloc do split
df = pd.read_csv(config.presence_csv)
dr = pd.read_csv(Path(config.fgadr_path) / 'DR_Seg_Grading_Label.csv',
                 header=None, names=['filename', 'dr_grade'])
df = df.merge(dr, on='filename', how='left')
suffix = df['filename'].str.rsplit('_', n=1).str[1].str.split('.').str[0]
df = df[suffix.isin(set(config.allowed_suffixes))].reset_index(drop=True)
tag = ''.join(sorted(config.allowed_suffixes))
splits = json.loads((REPO / 'outputs' / f'cv_splits_{tag}.json').read_text())
test_df = df.iloc[splits['test']].reset_index(drop=True)
print(f'{len(test_df)} imagens de teste')
test_df[CLASSES].sum().rename('imagens com a lesão (presence csv)')

## Modelos (5 folds)

In [ ]:
ckpts = sorted(RUN_DIR.glob('model_best_fold*.pth'))
models = []
for p in ckpts:
    m = build_model(config)
    ck = torch.load(p, map_location='cpu', weights_only=False)
    m.load_state_dict(ck['model_state_dict'])
    models.append(m.eval().to(DEVICE))
print(f'{len(models)} modelos carregados:', [p.name for p in ckpts])

## Carregamento, predição e plots

In [ ]:
MASK_DIRS = {'HardExudate': 'HardExudate_Masks', 'Hemorrhage': 'Hemohedge_Masks',
             'SoftExudate': 'SoftExudate_Masks', 'Microaneurysms': 'Microaneurysms_Masks'}
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
# cor de cada classe nos overlays (RGB)
COLORS = {'HardExudate': (255, 215, 0), 'Hemorrhage': (230, 30, 60),
          'SoftExudate': (0, 190, 255), 'Microaneurysms': (60, 220, 60)}
FGADR = Path(config.fgadr_path)


def load_sample(fname):
    # replica a transform de teste do dataset.py: Resize(H, W) + Normalize(ImageNet)
    img = cv2.cvtColor(cv2.imread(str(FGADR / 'Original_Images' / fname)), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)
    gt = np.zeros((len(CLASSES), H, W), np.uint8)
    for c, cls in enumerate(CLASSES):
        p = FGADR / MASK_DIRS[cls] / fname
        if p.exists():
            m = (cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) > config.bin_threshold).astype(np.uint8)
            gt[c] = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
    x = torch.from_numpy(((img.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1))
    return img, gt, x


@torch.no_grad()
def predict(x, fold=FOLD):
    # média dos logits entre folds, depois sigmoid (igual ao ensemble do train.py)
    ms = models if fold is None else [models[fold]]
    xb = x[None].to(DEVICE)
    logits = sum(m(xb) for m in ms) / len(ms)
    return torch.sigmoid(logits)[0].float().cpu().numpy()     # [C, H, W]


def dice(pred, gt):
    s = pred.sum() + gt.sum()
    return np.nan if s == 0 else 2 * (pred & gt).sum() / s


def overlay(img, masks, alpha=0.55):
    out = img.astype(np.float32).copy()
    for c, cls in enumerate(CLASSES):
        m = masks[c].astype(bool)
        out[m] = (1 - alpha) * out[m] + alpha * np.array(COLORS[cls], np.float32)
    return out.clip(0, 255).astype(np.uint8)


def only(masks, c):
    # zera todas as classes menos a c (para o overlay de uma classe só)
    keep = np.zeros_like(masks)
    keep[c] = masks[c]
    return keep


def error_map(img, pred, gt):
    # fundo = imagem em cinza escurecida; TP verde, FP vermelho, FN azul
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) * 0.35
    out = np.repeat(g[..., None], 3, axis=2)
    out[pred & gt] = (0, 230, 0)
    out[pred & ~gt] = (255, 40, 40)
    out[~pred & gt] = (40, 120, 255)
    return out.clip(0, 255).astype(np.uint8)


def _legend(ax):
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=np.array(COLORS[c]) / 255, label=k) for c, k in zip(CLASSES, CODES)],
              loc='lower right', fontsize=7, framealpha=0.7)


def show(fname, per_class=True, thr=THR, fold=FOLD):
    img, gt, x = load_sample(fname)
    prob = predict(x, fold)
    pred = prob > thr
    gtb = gt.astype(bool)
    ds = {k: dice(pred[c], gtb[c]) for c, k in enumerate(CODES)}
    tag = ' '.join(f'{k}={v:.2f}' if not np.isnan(v) else f'{k}=–' for k, v in ds.items())
    src = 'ensemble' if fold is None else f'fold{fold}'

    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    ax[0].imshow(img); ax[0].set_title(fname)
    ax[1].imshow(overlay(img, gtb)); ax[1].set_title('GT'); _legend(ax[1])
    ax[2].imshow(overlay(img, pred)); ax[2].set_title(f'pred ({src}, thr={thr})\nDice: {tag}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

    if per_class:
        fig, ax = plt.subplots(len(CLASSES), 3, figsize=(12, 4 * len(CLASSES)))
        for c, (cls, k) in enumerate(zip(CLASSES, CODES)):
            n_gt, n_pr = gtb[c].sum(), pred[c].sum()
            ax[c, 0].imshow(overlay(img, only(gtb, c)))
            ax[c, 0].set_title(f'{cls} GT ({n_gt} px)')
            im = ax[c, 1].imshow(prob[c], cmap='magma', vmin=0, vmax=1)
            ax[c, 1].set_title(f'prob (max {prob[c].max():.2f})')
            plt.colorbar(im, ax=ax[c, 1], fraction=0.046)
            ax[c, 2].imshow(error_map(img, pred[c], gtb[c]))
            d = ds[k]
            ax[c, 2].set_title(f'TP verde / FP verm. / FN azul\n'
                               f'{n_pr} px pred, Dice {"–" if np.isnan(d) else f"{d:.3f}"}')
            for a in ax[c]: a.axis('off')
        plt.tight_layout(); plt.show()
    return img, gt, prob


def show_zoom(fname, cls, size=128, thr=THR, fold=FOLD, out=None):
    # recorte centrado na região com mais GT ∪ pred da classe (útil p/ MA, que some em 512)
    if out is None:
        img, gt, x = load_sample(fname)
        out = img, gt, predict(x, fold)
    img, gt, prob = out
    c = CLASSES.index(cls)
    pred, g = prob[c] > thr, gt[c].astype(bool)
    dens = cv2.blur((pred | g).astype(np.float32), (size, size))
    if dens.max() == 0:
        print(f'{fname}: sem {cls} no GT nem na predição'); return
    cy, cx = np.unravel_index(dens.argmax(), dens.shape)
    y0 = int(np.clip(cy - size // 2, 0, H - size)); x0 = int(np.clip(cx - size // 2, 0, W - size))
    sl = (slice(y0, y0 + size), slice(x0, x0 + size))
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.3))
    ax[0].imshow(img[sl]); ax[0].set_title(f'{fname} [{y0}:{y0+size}, {x0}:{x0+size}]')
    ax[1].imshow(overlay(img, only(gt.astype(bool), c))[sl])
    ax[1].set_title(f'{cls} GT')
    ax[2].imshow(prob[c][sl], cmap='magma', vmin=0, vmax=1); ax[2].set_title('prob')
    ax[3].imshow(error_map(img, pred, g)[sl]); ax[3].set_title('TP / FP / FN')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

## Amostra aleatória do teste (re-execute para ver outras)

In [ ]:
N = 3
for fname in random.sample(list(test_df['filename']), N):
    show(fname, per_class=True)

## Uma imagem específica + zoom por classe

In [ ]:
fname = test_df['filename'].iloc[0]      # troque pelo nome que quiser, ex. '0563_3.png'
out = show(fname)
for cls in CLASSES:
    show_zoom(fname, cls, size=128, out=out)

## Dice por imagem em todo o teste (para achar os piores e melhores casos)

Roda os 89 × (5 folds) na GPU. `nan` = sem GT e sem predição daquela classe na imagem.

In [ ]:
rows = []
fnames = list(test_df['filename'])[:LIMIT]
for i, fname in enumerate(fnames):
    img, gt, x = load_sample(fname)
    pred = predict(x) > THR
    gtb = gt.astype(bool)
    r = {'filename': fname}
    for c, k in enumerate(CODES):
        r[f'dice_{k}'] = dice(pred[c], gtb[c])
        r[f'gt_px_{k}'] = int(gtb[c].sum())
        r[f'pred_px_{k}'] = int(pred[c].sum())
    rows.append(r)
per_img = pd.DataFrame(rows)
print('média por imagem (ignora nan):')
print(per_img[[f'dice_{k}' for k in CODES]].mean().round(3).to_string())
per_img.head()

In [ ]:
# piores imagens de uma classe (só onde há GT daquela classe)
K = 'hem'          # ex | hem | se | ma
n = 3
worst = per_img[per_img[f'gt_px_{K}'] > 0].sort_values(f'dice_{K}').head(n)
display(worst[['filename', f'dice_{K}', f'gt_px_{K}', f'pred_px_{K}']])
for fname in worst['filename']:
    out = show(fname, per_class=False)
    show_zoom(fname, CLASSES[CODES.index(K)], size=160, out=out)

In [ ]:
# falsos positivos puros: imagens SEM GT da classe onde o modelo pintou algo
K = 'se'
fp_only = per_img[(per_img[f'gt_px_{K}'] == 0) & (per_img[f'pred_px_{K}'] > 0)] \
    .sort_values(f'pred_px_{K}', ascending=False)
print(f'{len(fp_only)} imagens com FP de {K} e sem GT')
for fname in fp_only['filename'].head(2):
    out = show(fname, per_class=False)
    show_zoom(fname, CLASSES[CODES.index(K)], size=160, out=out)

## O HiLo está fazendo alguma coisa?

1. Escala efetiva por skip: `|α| · ||proj_out.weight||` (Frobenius), por fold. No DDR, com a receita
   do paper, ela colapsou para ~0.003.
2. Mapa do resíduo que o HiLo soma na skip: `|out − x|` médio nos canais, dividido por `|x|` médio,
   capturado com hook numa imagem. Mostra onde, espacialmente, a skip está sendo alterada.

In [ ]:
scale = pd.DataFrame({
    f'fold{i}': {f'skip{k}': (m.wavelet_modules[k].alpha.abs() *
                               m.wavelet_modules[k].proj_out.weight.norm()).item()
                 for k in m.wavelet_modules}
    for i, m in enumerate(models)})
scale['alpha_fold0'] = [models[0].wavelet_modules[k[4:]].alpha.item() for k in scale.index]
scale.round(4)

In [ ]:
def hilo_residuals(fname, fold=0):
    # hook em cada HiLoWaveletSkip: guarda |out - x| e |x| médios nos canais
    m = models[fold]
    store, hooks = {}, []
    for k, mod in m.wavelet_modules.items():
        def hook(mod, inp, out, k=k):
            x = inp[0]
            store[k] = ((out - x).abs().mean(1)[0].float().cpu().numpy(),
                        x.abs().mean(1)[0].float().cpu().numpy())
        hooks.append(mod.register_forward_hook(hook))
    img, gt, x = load_sample(fname)
    try:
        with torch.no_grad():
            m(x[None].to(DEVICE))
    finally:
        for h in hooks: h.remove()

    fig, ax = plt.subplots(1, len(store) + 1, figsize=(4 * (len(store) + 1), 4.2))
    ax[0].imshow(overlay(img, gt.astype(bool))); ax[0].set_title(f'{fname} (GT)'); _legend(ax[0])
    for a, (k, (d, xm)) in zip(ax[1:], store.items()):
        rel = d.mean() / (xm.mean() + 1e-8)
        a.imshow(d, cmap='viridis'); a.set_title(f'skip{k} {d.shape[0]}x{d.shape[1]}\n|Δ|/|x| = {rel:.4f}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()


hilo_residuals(random.choice(list(test_df['filename'])), fold=0)